# LeetCode 2138 — Divide a String Into Groups of Size k bằng Python

**Link đề:** https://leetcode.com/problems/divide-a-string-into-groups-of-size-k/
**Độ khó:** Easy · **Chủ đề:** String, Simulation

```python
class Solution:
    def divideString(self, s: str, k: int, fill: str) -> list[str]:
        ...
```

## Mục tiêu học
1. Hiểu bài "cắt chuỗi thành nhóm dài `k`" và cách **đệm (padding)** nhóm cuối bằng ký tự `fill`.
2. Nắm **slicing** `s[a:b]`, `range(start, stop, step)`, phép nhân chuỗi `"x" * 3`, phép `%` và chia trần.
3. Đi từ cách mô phỏng từng ký tự → cắt lát theo bước `k` → **đệm trước rồi cắt** (một công thức gọn).
4. Trace từng nhóm; chứng minh "bỏ fill rồi nối lại được đúng `s`".
5. Nhận dạng pattern "chunking" (chia khối) để dùng ở các bài khác.

## 1. Đề bài

Chia chuỗi `s` thành các nhóm, mỗi nhóm đúng `k` ký tự:
- Nhóm 1 = `k` ký tự đầu, nhóm 2 = `k` ký tự tiếp theo, ... Mỗi ký tự thuộc đúng một nhóm.
- Nhóm **cuối** nếu thiếu ký tự thì **đệm thêm** ký tự `fill` cho đủ `k`.

Yêu cầu: nếu bỏ phần `fill` ở nhóm cuối rồi nối các nhóm lại, ta phải được đúng `s`.

| `s` | `k` | `fill` | Kết quả |
|---|:-:|:-:|---|
| `"abcdefghi"` | 3 | `x` | `["abc","def","ghi"]` (chia vừa khít) |
| `"abcdefghij"` | 3 | `x` | `["abc","def","ghi","jxx"]` (nhóm cuối chỉ có `j`, đệm 2 `x`) |

**Constraints:** `1 <= len(s) <= 100`, `s` chỉ gồm chữ thường; `1 <= k <= 100`; `fill` là một chữ thường.

Chú ý: `k` có thể **lớn hơn** `len(s)` → chỉ có 1 nhóm, đệm rất nhiều `fill`.

In [1]:
s, k, fill = "abcdefghij", 3, "x"
print("len(s) =", len(s), "| k =", k)
print("số nhóm đủ k:", len(s) // k, "| ký tự dư:", len(s) % k)

len(s) = 10 | k = 3
số nhóm đủ k: 3 | ký tự dư: 1


## 2. Nền tảng Python cần biết

### 2.1. Chỉ số và slicing `s[a:b]`

- `s[i]` là ký tự thứ `i` (đếm từ 0).
- `s[a:b]` lấy các ký tự từ `a` đến `b - 1` (**không gồm `b`**), độ dài `b - a`.
- Slicing **không báo lỗi** khi `b` vượt quá độ dài: nó chỉ lấy tới hết chuỗi. Đây là tính chất ta sẽ khai thác.

In [2]:
s = "abcdefghij"
print(s[0], s[3])
print(s[0:3], s[3:6], s[6:9])
print(repr(s[9:12]), "<- chỉ còn 1 ký tự, không lỗi")
print(repr(s[20:23]), "<- hoàn toàn vượt biên: chuỗi rỗng")

a d
abc def ghi
'j' <- chỉ còn 1 ký tự, không lỗi
'' <- hoàn toàn vượt biên: chuỗi rỗng


### 2.2. `range(start, stop, step)` — nhảy theo bước

`range(0, len(s), k)` sinh các **điểm bắt đầu nhóm**: `0, k, 2k, ...` nhỏ hơn `len(s)`.

In [3]:
s, k = "abcdefghij", 3
starts = list(range(0, len(s), k))
print("điểm bắt đầu:", starts)
for i in starts:
    print(i, "->", s[i:i + k])

điểm bắt đầu: [0, 3, 6, 9]
0 -> abc
3 -> def
6 -> ghi
9 -> j


### 2.3. Nhân chuỗi và nối chuỗi

- `"x" * 3` → `"xxx"`; `"x" * 0` → `""` (chuỗi rỗng — rất tiện, không cần `if`).
- `a + b` nối hai chuỗi.

In [4]:
print(repr("x" * 3), repr("x" * 0))
part = "j"
print(part + "x" * (3 - len(part)))

'xxx' ''
jxx


### 2.4. `%` (chia lấy dư) và chia trần

- `len(s) % k` = số ký tự **dư** ở nhóm cuối (0 nếu chia vừa).
- Số nhóm = ⌈len(s) / k⌉ = `(len(s) + k - 1) // k` (chia **trần** chỉ bằng số nguyên).
- Số `fill` cần thêm = `(k - len(s) % k) % k`. Cái `% k` bên ngoài biến trường hợp "dư 0" thành "thêm 0" thay vì "thêm k".
- Trong Python còn có thể viết `(-len(s)) % k` — cùng kết quả vì `%` của Python luôn trả số không âm khi `k > 0`.

In [5]:
k = 3
for L in range(7, 13):
    groups = (L + k - 1) // k
    pad1 = (k - L % k) % k
    pad2 = (-L) % k
    print(f"len={L:>2}: dư={L % k}, số nhóm={groups}, cần đệm={pad1} (cách 2: {pad2})")

len= 7: dư=1, số nhóm=3, cần đệm=2 (cách 2: 2)
len= 8: dư=2, số nhóm=3, cần đệm=1 (cách 2: 1)
len= 9: dư=0, số nhóm=3, cần đệm=0 (cách 2: 0)
len=10: dư=1, số nhóm=4, cần đệm=2 (cách 2: 2)
len=11: dư=2, số nhóm=4, cần đệm=1 (cách 2: 1)
len=12: dư=0, số nhóm=4, cần đệm=0 (cách 2: 0)


### 2.5. Hàm chuỗi `ljust`

`t.ljust(width, ch)` trả về `t` được đệm thêm `ch` **bên phải** cho đủ `width`. Nếu `t` đã đủ dài thì giữ nguyên.

In [6]:
print("j".ljust(3, "x"))
print("abc".ljust(3, "x"))
print("ab".ljust(5, "z"))

jxx
abc
abzzz


### 2.6. List: `append` và list comprehension

- `result.append(x)` thêm `x` vào cuối list.
- `[biểu_thức for i in dãy]` tạo list mới gọn trong một dòng.

In [7]:
result = []
for i in range(3):
    result.append(str(i) * 2)
print(result)
print([str(i) * 2 for i in range(3)])

['00', '11', '22']
['00', '11', '22']


## 3. Cách 1 — Trực giác: mô phỏng từng ký tự

Người mới hay làm như cách ta chia bằng tay: có một "nhóm hiện tại" `cur`. Lần lượt thêm từng ký tự vào `cur`; khi `cur` đủ `k` ký tự thì cất vào kết quả và bắt đầu nhóm mới. Hết chuỗi mà `cur` còn dở dang → đệm `fill` rồi cất.

In [8]:
def divide_simulate(s: str, k: int, fill: str) -> list[str]:
    result = []
    cur = ""
    for ch in s:
        cur += ch
        if len(cur) == k:          # nhóm đã đủ
            result.append(cur)
            cur = ""
    if cur:                         # còn nhóm dở dang
        while len(cur) < k:
            cur += fill
        result.append(cur)
    return result

print(divide_simulate("abcdefghi", 3, "x"))
print(divide_simulate("abcdefghij", 3, "x"))
print(divide_simulate("ab", 5, "z"))

['abc', 'def', 'ghi']
['abc', 'def', 'ghi', 'jxx']
['abzzz']


### Đánh giá cách 1

- Đúng, dễ hiểu, `O(n + k)` (n = `len(s)`): mỗi ký tự thêm một lần, phần đệm tối đa `k - 1` ký tự.
- Nhược điểm nhỏ: `cur += ch` tạo chuỗi mới mỗi lần (chuỗi Python là **bất biến — immutable**), nên mỗi nhóm thực ra tốn `O(k²)` sao chép trong trường hợp xấu. Với `n, k ≤ 100` không đáng kể, nhưng là thói quen không tốt.
- Dài dòng: phải xử lý riêng "nhóm dở dang".

Hai ý tưởng giúp gọn hơn:
1. Cắt cả khối bằng **slicing** thay vì từng ký tự.
2. Xử lý nhóm cuối bằng **một công thức** thay cho vòng `while`.

In [9]:
t = "ab"
print(id(t))
t += "c"
print(id(t), "<- id khác: += tạo chuỗi mới, không sửa chuỗi cũ")

139709392407408
139708957366416 <- id khác: += tạo chuỗi mới, không sửa chuỗi cũ


## 4. Nút thắt và insight

**Nút thắt:** việc "gom từng ký tự" và "nhóm dở dang" làm code dài, dễ sai biên.

**Insight 1 — slicing:** nhóm thứ `g` (đếm từ 0) chính là `s[g*k : g*k + k]`. Slicing tự cắt ngắn ở cuối chuỗi, nên nhóm cuối là phần còn lại mà **không lỗi**.

**Insight 2 — đệm:** chỉ nhóm cuối có thể ngắn hơn `k`. Đệm nó bằng `fill * (k - len(nhom))` hoặc `ljust(k, fill)`.

**Insight 3 — đệm trước:** nếu ta đệm `s` thành độ dài chia hết cho `k` **ngay từ đầu**, mọi nhóm đều đủ `k` và không còn trường hợp đặc biệt nào.

In [10]:
s, k, fill = "abcdefghij", 3, "x"
padded = s + fill * ((-len(s)) % k)
print(padded, len(padded), len(padded) % k)
print([padded[i:i + k] for i in range(0, len(padded), k)])

abcdefghijxx 12 0
['abc', 'def', 'ghi', 'jxx']


## 5. Cách 2 — Lời giải chính: slicing theo bước `k` + đệm nhóm cuối

Giảng từng đoạn:

**Đoạn 1 — điểm bắt đầu:** `for i in range(0, len(s), k)`.

**Đoạn 2 — cắt:** `group = s[i:i + k]` (nhóm cuối có thể ngắn).

**Đoạn 3 — đệm:** `group += fill * (k - len(group))` — với nhóm đủ thì `k - len(group) = 0`, nhân ra chuỗi rỗng, không đổi gì. Không cần `if`!

**Đoạn 4 — cất:** `result.append(group)`.

In [11]:
def divide_slice(s: str, k: int, fill: str) -> list[str]:
    result = []
    for i in range(0, len(s), k):          # Đoạn 1
        group = s[i:i + k]                 # Đoạn 2
        group += fill * (k - len(group))   # Đoạn 3
        result.append(group)               # Đoạn 4
    return result

print(divide_slice("abcdefghi", 3, "x"))
print(divide_slice("abcdefghij", 3, "x"))
print(divide_slice("a", 1, "x"))
print(divide_slice("ab", 5, "z"))

['abc', 'def', 'ghi']
['abc', 'def', 'ghi', 'jxx']
['a']
['abzzz']


## 6. TRACE từng bước

`s = "abcdefghij"`, `k = 3`, `fill = "x"` (n = 10).

| Lượt | `i` | `s[i:i+3]` | `len` | Cần đệm `3 - len` | Nhóm cất vào | `result` sau |
|:-:|:-:|---|:-:|:-:|---|---|
| 1 | 0 | `abc` | 3 | 0 | `abc` | `[abc]` |
| 2 | 3 | `def` | 3 | 0 | `def` | `[abc, def]` |
| 3 | 6 | `ghi` | 3 | 0 | `ghi` | `[abc, def, ghi]` |
| 4 | 9 | `j` | 1 | 2 | `jxx` | `[abc, def, ghi, jxx]` |

Vòng dừng vì `i = 12` không còn `< 10`.

In [12]:
def trace_divide(s, k, fill):
    result = []
    print(f"s={s!r}, n={len(s)}, k={k}, fill={fill!r}")
    print("range ->", list(range(0, len(s), k)))
    for turn, i in enumerate(range(0, len(s), k), start=1):
        group = s[i:i + k]
        need = k - len(group)
        padded = group + fill * need
        result.append(padded)
        print(f"Lượt {turn}: i={i:>2}, s[{i}:{i + k}]={group!r:<6} len={len(group)}, đệm {need} -> {padded!r}, result={result}")
    return result

trace_divide("abcdefghij", 3, "x")

s='abcdefghij', n=10, k=3, fill='x'
range -> [0, 3, 6, 9]
Lượt 1: i= 0, s[0:3]='abc'  len=3, đệm 0 -> 'abc', result=['abc']
Lượt 2: i= 3, s[3:6]='def'  len=3, đệm 0 -> 'def', result=['abc', 'def']
Lượt 3: i= 6, s[6:9]='ghi'  len=3, đệm 0 -> 'ghi', result=['abc', 'def', 'ghi']
Lượt 4: i= 9, s[9:12]='j'    len=1, đệm 2 -> 'jxx', result=['abc', 'def', 'ghi', 'jxx']


['abc', 'def', 'ghi', 'jxx']

In [13]:
trace_divide("hello", 8, "z")   # k > len(s): chỉ 1 nhóm, đệm nhiều

s='hello', n=5, k=8, fill='z'
range -> [0]
Lượt 1: i= 0, s[0:8]='hello' len=5, đệm 3 -> 'hellozzz', result=['hellozzz']


['hellozzz']

## 7. Code nộp LeetCode — Cách 2

In [14]:
class Solution:
    def divideString(self, s: str, k: int, fill: str) -> list[str]:
        result = []
        for i in range(0, len(s), k):
            group = s[i:i + k]
            result.append(group + fill * (k - len(group)))
        return result

sol = Solution()
print(sol.divideString("abcdefghi", 3, "x"))
print(sol.divideString("abcdefghij", 3, "x"))

['abc', 'def', 'ghi']
['abc', 'def', 'ghi', 'jxx']


## 8. Cách 3 — Đệm trước rồi cắt (một dòng)

Tính số ký tự cần đệm `pad = (-len(s)) % k` (= `(k - len(s) % k) % k`), nối vào `s`, khi đó độ dài chia hết cho `k` và mọi lát cắt đều đủ `k`:

```python
s += fill * ((-len(s)) % k)
return [s[i:i + k] for i in range(0, len(s), k)]
```

Ưu điểm: không còn "nhóm cuối đặc biệt" trong vòng lặp — tư duy "**biến dữ liệu cho đẹp rồi xử lý đồng loạt**".

In [15]:
def divide_pad_first(s: str, k: int, fill: str) -> list[str]:
    s = s + fill * ((-len(s)) % k)
    return [s[i:i + k] for i in range(0, len(s), k)]

for args in [("abcdefghi", 3, "x"), ("abcdefghij", 3, "x"), ("ab", 5, "z"), ("abcd", 2, "q")]:
    print(args, "->", divide_pad_first(*args))

('abcdefghi', 3, 'x') -> ['abc', 'def', 'ghi']
('abcdefghij', 3, 'x') -> ['abc', 'def', 'ghi', 'jxx']
('ab', 5, 'z') -> ['abzzz']
('abcd', 2, 'q') -> ['ab', 'cd']


In [16]:
class Solution:
    def divideString(self, s: str, k: int, fill: str) -> list[str]:
        s += fill * (-len(s) % k)
        return [s[i:i + k] for i in range(0, len(s), k)]

print(Solution().divideString("abcdefghij", 3, "x"))

['abc', 'def', 'ghi', 'jxx']


Lưu ý thứ tự ưu tiên: `-len(s) % k` được hiểu là `(-len(s)) % k` vì dấu trừ một ngôi (unary minus) ưu tiên cao hơn `%`. Kiểm chứng:

In [17]:
L, k = 10, 3
print(-L % k, (-L) % k, -(L % k))

2 2 -1


## 9. Cách 4 — Tính theo số nhóm, dùng `ljust`

Biết trước số nhóm `g = ceil(n / k) = (n + k - 1) // k`, nhóm thứ `t` bắt đầu tại `t * k`. `ljust` lo phần đệm.

In [18]:
def divide_ljust(s: str, k: int, fill: str) -> list[str]:
    groups = (len(s) + k - 1) // k
    return [s[t * k:(t + 1) * k].ljust(k, fill) for t in range(groups)]

print(divide_ljust("abcdefghij", 3, "x"))
print(divide_ljust("ab", 5, "z"))

['abc', 'def', 'ghi', 'jxx']
['abzzz']


**Bonus (để biết, không khuyến khích khi phỏng vấn):** `textwrap.wrap(s, k)` của thư viện chuẩn cũng chia được chuỗi *không có khoảng trắng* thành các mẩu dài `k`, nhưng nó được thiết kế cho văn bản (sẽ xử lý khoảng trắng khác hẳn), nên dễ gây bất ngờ.

In [19]:
import textwrap
print([w.ljust(3, "x") for w in textwrap.wrap("abcdefghij", 3)])
print(textwrap.wrap("ab cd", 2), "<- khoảng trắng bị bỏ: hành vi khác hẳn đề bài")

['abc', 'def', 'ghi', 'jxx']
['ab', 'cd'] <- khoảng trắng bị bỏ: hành vi khác hẳn đề bài


## 10. Vì sao đúng? (invariant)

Gọi `n = len(s)`. Các điểm bắt đầu là `0, k, 2k, ..., (g-1)k` với `g = ⌈n/k⌉`.

- **Phủ kín, không chồng lấn:** lát `[tk, tk + k)` nối tiếp lát `[(t-1)k, tk)`, nên mỗi chỉ số `0..n-1` thuộc đúng một lát.
- **Đúng thứ tự:** các lát được tạo theo `t` tăng dần.
- **Chỉ nhóm cuối có thể thiếu:** với `t < g-1` ta có `tk + k ≤ (g-1)k < n` (vì `g = ⌈n/k⌉` nên `(g-1)k < n`), nên lát đủ `k`. Chỉ lát cuối có thể bị slicing cắt ngắn còn `n - (g-1)k` ký tự (từ 1 đến k).
- **Đệm đúng số lượng:** nhóm cuối được thêm `k - len` ký tự `fill`, nên mọi nhóm dài đúng `k`.
- **Khôi phục được `s`:** bỏ các `fill` thêm vào (chỉ nằm ở cuối nhóm cuối) rồi nối các lát theo thứ tự → đúng `s[0:n]`.

Cách 3 tương đương: sau khi đệm, độ dài là `g·k`, mọi lát đủ `k`, và phần đệm chỉ nằm ở cuối.

## 11. Độ phức tạp

Gọi `n = len(s)`, `g = ⌈n/k⌉` số nhóm. Tổng độ dài đầu ra là `g·k < n + k`.

| Cách | Time | Space (kể cả output) | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng từng ký tự | `O(n·k)` worst do `cur += ch` sao chép; thực tế CPython thường tối ưu | `O(n + k)` | dài, nhiều nhánh |
| 2. Slicing + đệm nhóm cuối | `O(n + k)` | `O(n + k)` | lời giải chính |
| 3. Đệm trước rồi cắt | `O(n + k)` | `O(n + k)` | gọn nhất |
| 4. Số nhóm + `ljust` | `O(n + k)` | `O(n + k)` | tương đương cách 2 |

Không cách nào nhanh hơn `O(n + k)` được vì riêng việc **ghi ra đáp án** đã tốn chừng ấy ký tự.

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

Nhóm test:
1. Hai ví dụ của đề.
2. `k = 1` (mỗi ký tự một nhóm, không đệm).
3. `k = len(s)` (một nhóm, không đệm) và `k > len(s)` (một nhóm, đệm nhiều).
4. Dư đúng 1 và dư đúng `k-1`.
5. `fill` trùng với ký tự trong `s` — đầu ra vẫn đúng.

In [20]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<26} PASS {actual}")

impls = [divide_simulate, divide_slice, divide_pad_first, divide_ljust, Solution().divideString]
cases = [
    ("đề ví dụ 1", ("abcdefghi", 3, "x"), ["abc", "def", "ghi"]),
    ("đề ví dụ 2", ("abcdefghij", 3, "x"), ["abc", "def", "ghi", "jxx"]),
    ("k = 1", ("abc", 1, "x"), ["a", "b", "c"]),
    ("k = len(s)", ("abc", 3, "x"), ["abc"]),
    ("k > len(s)", ("ab", 5, "z"), ["abzzz"]),
    ("dư k-1", ("abcde", 3, "x"), ["abc", "dex"]),
    ("dư 1", ("abcd", 3, "x"), ["abc", "dxx"]),
    ("fill trùng ký tự", ("aaaa", 3, "a"), ["aaa", "aaa"]),
    ("len 1", ("q", 4, "w"), ["qwww"]),
]
for name, args, exp in cases:
    for f in impls:
        assert f(*args) == exp, (name, f)
    check(name, divide_slice(*args), exp)
print("Mọi cách cho cùng kết quả!")

đề ví dụ 1                 PASS ['abc', 'def', 'ghi']
đề ví dụ 2                 PASS ['abc', 'def', 'ghi', 'jxx']
k = 1                      PASS ['a', 'b', 'c']
k = len(s)                 PASS ['abc']
k > len(s)                 PASS ['abzzz']
dư k-1                     PASS ['abc', 'dex']
dư 1                       PASS ['abc', 'dxx']
fill trùng ký tự           PASS ['aaa', 'aaa']
len 1                      PASS ['qwww']
Mọi cách cho cùng kết quả!


### Kiểm thử ngẫu nhiên + kiểm tra tính chất

Ngoài so sánh các cách với nhau, ta kiểm tra **tính chất** của đáp án (property-based):
- mọi nhóm dài đúng `k`;
- nối lại rồi cắt bỏ phần đệm cuối (`[:n]`) được `s`;
- phần sau vị trí `n` toàn là `fill`, và số nhóm = ⌈n/k⌉.

In [21]:
import random, string
rng = random.Random(2138)
for _ in range(500):
    n = rng.randint(1, 30)
    k = rng.randint(1, 35)
    s = "".join(rng.choice("abc") for _ in range(n))
    fill = rng.choice(string.ascii_lowercase)
    out = divide_slice(s, k, fill)
    joined = "".join(out)
    assert all(len(g) == k for g in out)
    assert joined[:n] == s
    assert set(joined[n:]) <= {fill}
    assert len(out) == (n + k - 1) // k
    for f in impls:
        assert f(s, k, fill) == out
print("500 test ngẫu nhiên: PASS")

500 test ngẫu nhiên: PASS


## 13. Lỗi người mới hay gặp

### Lỗi 1: Đệm `k` ký tự khi chia vừa khít

In [22]:
def wrong_pad(s, k, fill):
    s = s + fill * (k - len(s) % k)       # thiếu % k bên ngoài
    return [s[i:i + k] for i in range(0, len(s), k)]

print("sai :", wrong_pad("abcdef", 3, "x"))
print("đúng:", divide_pad_first("abcdef", 3, "x"))

sai : ['abc', 'def', 'xxx']
đúng: ['abc', 'def']


**Vì sao sai:** khi `len(s) % k == 0`, `k - 0 = k` → thêm cả một nhóm toàn `fill`. **Sửa:** `(k - len(s) % k) % k` hoặc `(-len(s)) % k`.

### Lỗi 2: Dùng `range(len(s) // k)` → mất nhóm cuối

In [23]:
def wrong_floor(s, k, fill):
    return [s[t * k:(t + 1) * k] for t in range(len(s) // k)]

print("sai :", wrong_floor("abcdefghij", 3, "x"))
print("đúng:", divide_slice("abcdefghij", 3, "x"))

sai : ['abc', 'def', 'ghi']
đúng: ['abc', 'def', 'ghi', 'jxx']


**Vì sao sai:** `10 // 3 = 3` là chia **sàn** — bỏ mất nhóm dở dang `"j"`. **Sửa:** chia **trần** `(n + k - 1) // k`, hoặc dùng `range(0, n, k)`.

### Lỗi 3: Đánh chỉ số thủ công `s[i]` vượt biên

In [24]:
def wrong_index(s, k, fill):
    result = []
    for i in range(0, len(s), k):
        group = ""
        for j in range(k):
            group += s[i + j]            # IndexError khi nhóm cuối thiếu
        result.append(group)
    return result

try:
    wrong_index("abcdefghij", 3, "x")
except IndexError as e:
    print("IndexError:", e)

IndexError: string index out of range


**Vì sao sai:** truy cập từng chỉ số **không** "khoan dung" như slicing. **Sửa:** kiểm tra `i + j < len(s)` rồi mới đọc, nếu không thì dùng `fill` — hoặc đơn giản dùng slicing.

### Lỗi 4: Lấy phần dư bằng `s[-(n % k):]`
Ai viết "nhóm cuối = phần dư `s[-(n % k):]`" sẽ bị sai khi `n % k == 0` (vì `s[-0:]` = `s[0:]` = cả chuỗi).

In [25]:
s, k = "abcdef", 3
print(repr(s[-(len(s) % k):]), "<- tưởng là phần dư rỗng nhưng lại ra cả chuỗi!")

'abcdef' <- tưởng là phần dư rỗng nhưng lại ra cả chuỗi!


## 14. Mẹo nhận dạng pattern

- "Chia thành các khối kích thước `k`", "mỗi `k` phần tử", "chunk", "batch" → `range(0, n, k)` + slicing `[i:i+k]`.
- "Phần cuối thiếu thì đệm" → hoặc đệm từng khối bằng `fill * (k - len)`, hoặc **đệm trước** bằng `(-n) % k`.
- Số khối = chia trần `(n + k - 1) // k`. Thuộc lòng công thức này!
- Slicing không lỗi khi vượt biên; indexing thì có — chọn slicing để giảm nhánh `if`.

Bài liên quan: #1556 Thousand Separator, #482 License Key Formatting, #541 Reverse String II (xử lý theo khối `2k`), #2243 Calculate Digit Sum of a String.

## 15. Biến thể

### 15.1. Chunk cho list và **không** đệm

In [26]:
def chunks(items, k):
    return [items[i:i + k] for i in range(0, len(items), k)]

print(chunks([1, 2, 3, 4, 5, 6, 7], 3))

[[1, 2, 3], [4, 5, 6], [7]]


### 15.2. Đệm ở **bên trái** của nhóm đầu (căn phải), giống tách hàng nghìn
Ví dụ chia `"1234567"` thành nhóm 3 tính từ **phải**: `["1", "234", "567"]`.

In [27]:
def chunks_from_right(s, k):
    first = len(s) % k or k          # nhóm đầu dài len%k, nếu 0 thì dài k
    out = [s[:first]]
    out += [s[i:i + k] for i in range(first, len(s), k)]
    return out

print(chunks_from_right("1234567", 3))
print(".".join(chunks_from_right("123456", 3)))

['1', '234', '567']
123.456


Ở đây `len(s) % k or k` dùng toán tử `or`: nếu vế trái là `0` (falsy) thì lấy vế phải `k`.

### 15.3. Generator: cắt khối "lười" (lazy) bằng `yield`
Hữu ích khi dữ liệu rất lớn: không tạo cả list kết quả một lúc.

In [28]:
def iter_groups(s, k, fill):
    for i in range(0, len(s), k):
        g = s[i:i + k]
        yield g + fill * (k - len(g))

gen = iter_groups("abcdefg", 3, "_")
print(next(gen), next(gen), next(gen))
print(list(iter_groups("abcdefg", 3, "_")))

abc def g__
['abc', 'def', 'g__']


## 16. Bài tập (TODO — tự làm, đáp án ở cuối)

### Bài 1 — Đoán output (không chạy)
1. `"abcdefg"[5:9]` = ?
2. `list(range(0, 7, 3))` = ?
3. `(-7) % 3` = ?
4. `divideString("abcdefg", 3, "#")` = ?

### Bài 2 — Tính tay
Với `n = len(s) = 23`, `k = 5`: có bao nhiêu nhóm, nhóm cuối có mấy ký tự thật, cần bao nhiêu `fill`?

### Bài 3 — Điền chỗ trống (cách slicing)

In [29]:
def exercise_divide(s: str, k: int, fill: str) -> list[str]:
    result = []
    # TODO 1: duyệt i từ 0 tới len(s) theo bước k
    # TODO 2: cắt group = s[i:i + k]
    # TODO 3: đệm fill cho đủ k (không cần if)
    # TODO 4: append vào result
    return result

# assert exercise_divide("abcdefghij", 3, "x") == ["abc", "def", "ghi", "jxx"]

### Bài 4 — Phép ngược: từ danh sách nhóm và `fill`, khôi phục `s`
Biết rằng ký tự `fill` **không** xuất hiện trong `s` gốc.

In [30]:
def exercise_restore(groups: list[str], fill: str) -> str:
    # TODO: nối lại rồi bỏ fill ở cuối
    pass

# assert exercise_restore(["abc", "def", "ghi", "jxx"], "x") == "abcdefghij"

### Bài 5 — Biến thể: định dạng license key
Cho chuỗi chữ/số `s` (đã bỏ dấu gạch), chia nhóm `k` **từ bên phải**, viết hoa, nối bằng `-`. Ví dụ `"2k9a5c3"`, `k = 4` → `"2K9-A5C3"`.

In [31]:
def exercise_license_key(s: str, k: int) -> str:
    # TODO
    pass

# assert exercise_license_key("2k9a5c3", 4) == "2K9-A5C3"
# assert exercise_license_key("abcdef", 3) == "ABC-DEF"

### Bài 6 — Mini challenge: đảo ngược từng khối
Chia `s` thành nhóm `k` (không đệm), **đảo ngược** từng nhóm rồi nối lại. Ví dụ `"abcdefg"`, `k = 3` → `"cbafedg"`.

In [32]:
def exercise_reverse_each_block(s: str, k: int) -> str:
    # TODO
    pass

# assert exercise_reverse_each_block("abcdefg", 3) == "cbafedg"

### Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | Slicing không lỗi khi vượt biên | `%` của Python luôn không âm | nhóm cuối là `"g"` |
| 2 | Chia trần | `(23 + 5 - 1) // 5` | dư `23 % 5` |
| 3 | `range(0, len(s), k)` | `fill * (k - len(group))` | `result.append(group + ...)` |
| 4 | Fill chỉ ở cuối | `"".join(groups)` | `.rstrip(fill)` |
| 5 | Khối đầu dài `n % k` | `n % k or k` | `"-".join(...)`, `.upper()` |
| 6 | Mỗi khối dùng `[::-1]` | `range(0, len(s), k)` | `"".join(s[i:i+k][::-1] for i in ...)` |

## 17. Đáp án tham khảo (tự làm trước rồi hãy xem)

**Bài 1:** `"fg"`; `[0, 3, 6]`; `2`; `["abc", "def", "g##"]`.

**Bài 2:** số nhóm `(23 + 4) // 5 = 5`; nhóm cuối có `23 % 5 = 3` ký tự thật; cần `5 - 3 = 2` ký tự `fill` (kiểm: `(-23) % 5 = 2`).

In [33]:
print(repr("abcdefg"[5:9]), list(range(0, 7, 3)), (-7) % 3, Solution().divideString("abcdefg", 3, "#"))
print((23 + 4) // 5, 23 % 5, (-23) % 5)

'fg' [0, 3, 6] 2 ['abc', 'def', 'g##']
5 3 2


In [34]:
def answer_divide(s, k, fill):
    result = []
    for i in range(0, len(s), k):
        group = s[i:i + k]
        result.append(group + fill * (k - len(group)))
    return result


def answer_restore(groups, fill):
    return "".join(groups).rstrip(fill)


def answer_license_key(s, k):
    s = s.upper()
    first = len(s) % k or k
    parts = [s[:first]] + [s[i:i + k] for i in range(first, len(s), k)]
    return "-".join(parts)


def answer_reverse_each_block(s, k):
    return "".join(s[i:i + k][::-1] for i in range(0, len(s), k))


assert answer_divide("abcdefghij", 3, "x") == ["abc", "def", "ghi", "jxx"]
for name, args, exp in cases:
    assert answer_divide(*args) == exp, name
assert answer_restore(["abc", "def", "ghi", "jxx"], "x") == "abcdefghij"
assert answer_license_key("2k9a5c3", 4) == "2K9-A5C3"
assert answer_license_key("abcdef", 3) == "ABC-DEF"
assert answer_reverse_each_block("abcdefg", 3) == "cbafedg"
print("Đáp án tham khảo: PASS toàn bộ test!")

Đáp án tham khảo: PASS toàn bộ test!


Lưu ý về Bài 4: `rstrip(fill)` chỉ an toàn vì ta **giả sử** `fill` không có trong `s`. Nếu `s = "abx"`, `fill = "x"`, `k = 2` thì nhóm là `["ab", "xx"]` và `rstrip` xóa luôn chữ `x` thật → muốn khôi phục chính xác phải biết `n` gốc rồi lấy `joined[:n]`.

## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def divideString(self, s: str, k: int, fill: str) -> list[str]:
        s += fill * (-len(s) % k)                       # đệm cho chia hết
        return [s[i:i + k] for i in range(0, len(s), k)]  # cắt khối
```

**Khung tư duy chunking:** điểm bắt đầu `range(0, n, k)` → lát `[i:i+k]` → xử lý khối cuối (đệm/để nguyên) → số khối `(n + k - 1) // k`.

**Checklist trước khi nộp:**
- [ ] Chia vừa khít có bị đệm thừa một nhóm `fill` không? (`% k` bên ngoài)
- [ ] Có mất nhóm cuối do chia sàn không?
- [ ] `k > len(s)` có ra đúng 1 nhóm không?
- [ ] Mọi nhóm dài đúng `k`?
- [ ] Trả về `list[str]`, đúng thứ tự?